# Gemma-4 Benchmark: Base Model vs. Fine-Tune

İki modeli, iki benchmark ile karşılaştırır:

| Model | Ne |
|---|---|
| **Base** | `unsloth/gemma-4-E4B-it` — hiç eğitilmemiş hâli |
| **Fine-tune** | `bilalabic/gemma_4_math-toolcall-tr_lora` — bizim LoRA adaptörümüz |

| Benchmark | Ölçtüğü şey | Neden |
|---|---|---|
| **1. Türkçe MMLU** | Genel bilgi (çoktan seçmeli) | Fine-tune genel yeteneği **bozdu mu?** (catastrophic forgetting) |
| **2. Matematik Tool-Call** | Doğru aracı çağırma + gerekmiyorsa çağırmama | **Asıl hedefimiz** — iyileşme burada olmalı |

### Üç tasarım kararı

**1. Ollama değil Unsloth.** Fine-tune modelimiz GGUF olarak Ollama'da yok, HF'de LoRA
adaptörü olarak duruyor. Unsloth ikisini de doğrudan yükleyebiliyor.

**2. Tek yükleme, adaptörü aç/kapa.** LoRA = base ağırlıklar + küçük bir ek. Adaptörü
`disable_adapter()` ile kapatınca elimizde **tam olarak base model** kalır. Bu hem ~10 GB
bellek kazandırır hem de iki modelin **aynı kuantizasyonla** çalışmasını garanti eder —
yoksa karşılaştırma adil olmaz.

**3. Gerçek held-out test.** Model veri setinin **757 örneklik** hâliyle eğitildi; veri
seti şimdi **1.207**. Yani son **450 örnek modelin hiç görmediği** veridir. Matematik
benchmark'ı bunun üzerine kurulur — ezber değil, genelleme ölçülür.

> ⏱️ Tahmini süre (T4/L4): MMLU 250 soru ≈ 25 dk, matematik 150 örnek ≈ 20 dk (iki model için toplam).

## 1. Kurulum

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec sentence-transformers pandas matplotlib
import torch; torch._dynamo.config.recompile_limit = 64

## 2. Ayarlar

Buradaki sayıları değiştirerek testin kapsamını ve süresini ayarlarsın.

In [ ]:
# ============================ AYARLAR ============================
FINETUNE_REPO = "bilalabic/gemma_4_math-toolcall-tr_lora"   # LoRA adaptorumuz (base'i icinde tasir)
DATASET_REPO  = "bilalabic/math-toolcall-tr"                # kendi veri setimiz

# Kac soru test edilsin? Dusuk tut -> hizli calisir; yuksek tut -> sonuc daha guvenilir.
# MMLU tam veri seti binlerce soru; 250 makul bir denge.
MMLU_SORU_SAYISI = 250
MATEMATIK_ORNEK_SAYISI = 150

# Model 757 ornekle egitildi. Bu indeksten SONRAKI ornekleri model HIC gormedi.
# Matematik benchmark'i sadece bu gorulmemis kisim uzerinde calisir.
EGITIMDE_KULLANILAN = 757

MAX_SEQ_LENGTH = 2048   # modele verilebilecek en uzun girdi (token). Egitimde de 2048 kullanildi.
SEED = 42               # tekrarlanabilirlik icin sabit tohum
# =================================================================

import torch, time, json, re, gc
import pandas as pd
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK - Runtime > GPU sec!")

## 3. Modeli yükle

Tek bir yükleme yapıyoruz: LoRA adaptörü **base modeli de beraberinde** getirir.
Sonra adaptörü açıp kapatarak iki modeli de test edeceğiz.

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name     = FINETUNE_REPO,
    # dtype=None -> donanima gore otomatik secer (A100'de bfloat16, T4'te float16).
    dtype          = None,
    # Modelin isleyebilecegi en uzun dizi. Uzun tutmak bellek yer; 2048 bizim egitimle ayni.
    max_seq_length = MAX_SEQ_LENGTH,
    # 4-bit kuantizasyon: agirliklari 4 bite sikistirir. ~4x az bellek, cok az dogruluk kaybi.
    # Egitim de 4-bit yapildigi icin test de 4-bit olmali -> adil karsilastirma.
    load_in_4bit   = True,
)

# Modeli degerlendirme moduna al: dropout gibi egitime ozel katmanlari kapatir.
model.eval()
print("Model yuklendi. Adaptor aktif mi:", hasattr(model, "disable_adapter"))

## 4. Üretim fonksiyonu — parametreler ne işe yarıyor?

Benchmark'ta **belirlenimci (deterministic)** üretim kullanıyoruz. Sebep: aynı soruya
her çalıştırmada aynı cevabı vermeli ki iki model adil karşılaştırılsın ve sonuç
tekrarlanabilir olsun.

| Parametre | Ne yapar | Benchmark'ta neden böyle |
|---|---|---|
| `do_sample=False` | Rastgelelik yok, her adımda **en olası** token seçilir (greedy) | Aynı girdi → aynı çıktı. Ölçüm gürültüsü sıfır |
| `temperature` | Olasılıkları yayar/keskinleştirir. Yüksek = yaratıcı, düşük = tutarlı | `do_sample=False` iken **etkisiz**, o yüzden göndermiyoruz |
| `top_p` / `top_k` | Örnekleme havuzunu daraltır | Yine sadece `do_sample=True` iken anlamlı |
| `max_new_tokens` | En fazla kaç token üretilecek | MMLU'da kısa (tek harf yeter), tool-call'da uzun (JSON + açıklama) |
| `use_cache=True` | Önceki hesapları saklar (KV cache) | Üretimi belirgin biçimde hızlandırır |

> **Not:** Gemma-4'ün önerdiği `temperature=1.0, top_p=0.95, top_k=64` ayarları **sohbet**
> içindir. Benchmark'ta rastgelelik istemeyiz — bu yüzden greedy kullanıyoruz.

In [ ]:
@torch.no_grad()   # gradyan hesabini kapatir -> daha hizli, cok daha az bellek
def uret(mesaj: str, max_new_tokens: int = 64) -> str:
    """Modele bir kullanici mesaji verir, urettigi metni dondurur."""
    mesajlar = [{"role": "user", "content": [{"type": "text", "text": mesaj}]}]

    # Chat template'i uygular: mesaji modelin bekledigi <start_of_turn> formatina cevirir.
    girdi = tokenizer.apply_chat_template(
        mesajlar,
        add_generation_prompt = True,   # sona "model sirasi" isareti koyar -> model cevap yazmaya baslar
        tokenize    = True,
        return_dict = True,
        return_tensors = "pt",
    ).to("cuda")

    cikti = model.generate(
        **girdi,
        max_new_tokens = max_new_tokens,
        do_sample      = False,   # greedy -> belirlenimci
        use_cache      = True,
    )

    # Sadece YENI uretilen kismi al (girdi promptunu kes).
    yeni_tokenlar = cikti[0][girdi["input_ids"].shape[1]:]
    return tokenizer.decode(yeni_tokenlar, skip_special_tokens=True).strip()


# Adaptoru kapatip acmayi kolaylastiran yardimci.
from contextlib import contextmanager

# Guvenlik kontrolu: adaptor kapatilamiyorsa iki model de AYNI olur ve
# karsilastirma sessizce anlamsizlasir. Bunu bastan yakalayalim.
assert hasattr(model, "disable_adapter"), (
    "Model bir PeftModel degil - adaptor kapatilamiyor.\n"
    "Cozum: base'i ayrica yukle ->\n"
    "  base_model, _ = FastModel.from_pretrained('unsloth/gemma-4-E4B-it',\n"
    "                      max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)"
)

@contextmanager
def model_olarak(hangisi: str):
    """'base' -> LoRA kapali (saf temel model) | 'finetune' -> LoRA acik."""
    if hangisi == "base":
        with model.disable_adapter():   # adaptor agirliklarini gecici olarak devre disi birakir
            yield
    else:
        yield

# Hizli duman testi: iki model FARKLI cikti veriyor mu?
# Ikisi de birebir ayni ciktiyi veriyorsa adaptor gercekten devrede degil demektir.
_ciktilar = {}
for ad in ("base", "finetune"):
    with model_olarak(ad):
        _ciktilar[ad] = uret("12'nin asal çarpanlarını bul.", max_new_tokens=80)
    print(f"[{ad:9}] {_ciktilar[ad][:150]}\n")

if _ciktilar["base"] == _ciktilar["finetune"]:
    print("UYARI: iki cikti birebir ayni. Adaptor devrede olmayabilir - kontrol et!")
else:
    print("OK: adaptor davranisi degistiriyor, karsilastirma anlamli.")

## 5. Benchmark 1 — Türkçe MMLU

Çoktan seçmeli genel bilgi testi. Burada **iyileşme beklemiyoruz** — beklentimiz
fine-tune'un genel yeteneği *bozmamış* olması. Skor ciddi düştüyse model dar bir alana
aşırı uyum sağlamış (catastrophic forgetting) demektir.

Cevap kontrolü üç aşamalı: (1) doğrudan harf eşleşmesi, (2) "A)" gibi ön ekleri ayıklama,
(3) model harf yerine cümle yazdıysa **anlamsal benzerlik** ile en yakın şıkkı bulma.

In [ ]:
from sentence_transformers import SentenceTransformer

# Model harf yerine "cevap 12'dir" gibi yazarsa, hangi sikka en yakin oldugunu bulmak icin.
benzerlik_modeli = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

HARFLER = ['A', 'B', 'C', 'D', 'E']

def cevap_dogru_mu(dogru_index: int, verilen: str, secenekler: list) -> bool:
    dogru_harf = HARFLER[dogru_index]
    verilen = verilen.upper().strip()

    # 1) Tam harf eslesmesi
    if verilen == dogru_harf:
        return True

    # 2) "A)", "A:", "A -" gibi kaliplarin ilk harfini al
    if len(verilen) > 1 and verilen[1] in [" ", ":", ")", "=", "-", ".", ","]:
        return verilen[0] == dogru_harf

    # 3) Serbest metin -> anlamsal olarak en yakin sikki bul
    if not verilen:
        return False
    e_cevap = benzerlik_modeli.encode([verilen])
    e_secenek = benzerlik_modeli.encode(secenekler)
    puanlar = benzerlik_modeli.similarity(e_cevap, e_secenek).tolist()[0]
    return puanlar.index(max(puanlar)) == dogru_index


# Veri setini yukle (alibayram'in Turkce MMLU calismasi)
mmlu = pd.read_parquet(
    "hf://datasets/alibayram/yapay_zeka_turkce_mmlu_model_cevaplari/data/train-00000-of-00001.parquet"
)
# Sabit tohumla karistir -> her calistirmada AYNI alt kume secilir (tekrarlanabilirlik)
mmlu = mmlu.sample(n=min(MMLU_SORU_SAYISI, len(mmlu)), random_state=SEED).reset_index(drop=True)
print(f"MMLU: {len(mmlu)} soru | bolum sayisi: {mmlu['bolum'].nunique()}")

In [ ]:
def ilerleme(guncel, toplam, uzunluk=30):
    dolu = int(uzunluk * guncel / toplam)
    return f"[{'#' * dolu}{'-' * (uzunluk - dolu)}] {100 * guncel / toplam:5.1f}%"


def mmlu_calistir(model_adi: str) -> dict:
    dogru = 0
    bolum_dogru, bolum_toplam = {}, {}
    kayitlar = []
    basla = time.time()

    with model_olarak(model_adi):
        for i in range(len(mmlu)):
            satir = mmlu.iloc[i]

            # Soruyu ve siklari tek metne topla
            metin = satir['soru'] + "\n"
            for j, secenek in enumerate(satir['secenekler']):
                metin += f"{HARFLER[j]}: {secenek}\n"

            prompt = ("Sana soru ve seçenekleri veriyorum. Sadece hangi seçeneğin doğru "
                      "cevap olduğunu yaz. Örneğin 'A' veya 'B' gibi. Açıklama yapma!\n"
                      "Soru: " + metin)

            # max_new_tokens=16: tek harf bekliyoruz, uzun uretim sadece zaman kaybi olur
            cevap = uret(prompt, max_new_tokens=16)
            sonuc = cevap_dogru_mu(satir['cevap'], cevap, list(satir['secenekler']))

            bolum = satir['bolum']
            bolum_toplam[bolum] = bolum_toplam.get(bolum, 0) + 1
            if sonuc:
                dogru += 1
                bolum_dogru[bolum] = bolum_dogru.get(bolum, 0) + 1

            kayitlar.append({"soru_no": i, "bolum": bolum, "model": model_adi,
                             "cevap": cevap, "dogru_mu": sonuc})

            print(f"\r[{model_adi:9}] {ilerleme(i+1, len(mmlu))} "
                  f"dogru {dogru}/{i+1} = {dogru/(i+1)*100:5.1f}% "
                  f"({time.time()-basla:.0f}s)", end="")
    print()

    return {
        "model": model_adi,
        "dogru": dogru,
        "toplam": len(mmlu),
        "basari": round(100 * dogru / len(mmlu), 2),
        "sure_sn": round(time.time() - basla, 1),
        "bolum_basari": {b: round(100 * bolum_dogru.get(b, 0) / t, 1) for b, t in bolum_toplam.items()},
        "kayitlar": kayitlar,
    }


mmlu_sonuc = {ad: mmlu_calistir(ad) for ad in ("base", "finetune")}
for ad, s in mmlu_sonuc.items():
    print(f"{ad:9} -> %{s['basari']}  ({s['dogru']}/{s['toplam']}, {s['sure_sn']}s)")

## 6. Benchmark 2 — Matematik Tool-Call

**Asıl ölçmek istediğimiz.** Modele matematik fonksiyonları sunuyoruz ve bakıyoruz:

| Metrik | Ne ölçer |
|---|---|
| **Format geçerliliği** | `<tool_call>{...}</tool_call>` düzgün JSON mu? |
| **Araç seçimi** | Doğru fonksiyonu mu çağırdı? |
| **Çekimserlik (abstain)** | Araç *gerekmediğinde* çağırmamayı biliyor mu? |
| **Genel doğruluk** | Yukarıdakilerin birleşimi |

Test verisi: modelin **hiç görmediği** son 450 örnek (eğitim 757'de kesildi).

In [ ]:
from datasets import load_dataset

# Ham veri setini indir (tools ve etiketler burada; sharegpt surumunde yok)
_ds = load_dataset(DATASET_REPO, split="train")
print("HF'deki toplam ornek:", len(_ds))

# GORULMEMIS kisim: egitim 757'de kesildi, sonrasi model icin tamamen yeni.
gorulmemis = _ds.select(range(EGITIMDE_KULLANILAN, len(_ds)))
print("Gorulmemis ornek:", len(gorulmemis))

# Tekrarlanabilir alt kume
gorulmemis = gorulmemis.shuffle(seed=SEED).select(range(min(MATEMATIK_ORNEK_SAYISI, len(gorulmemis))))
print("Test edilecek:", len(gorulmemis))

In [ ]:
# --- Modelin ciktisindan arac cagrilarini ayikla ---
TOOL_CALL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)

def cagrilari_ayikla(metin: str):
    """Metindeki <tool_call> bloklarini parse eder.
    Doner: (cagri_listesi, format_gecerli_mi)"""
    ham = TOOL_CALL_RE.findall(metin)
    if not ham:
        return [], True          # hic cagri yok -> format acisindan sorun degil
    cagrilar = []
    for parca in ham:
        try:
            cagrilar.append(json.loads(parca))
        except json.JSONDecodeError:
            return cagrilar, False   # bozuk JSON uretmis
    return cagrilar, True


# Egitim verisindeki kullanici mesajini geri cikar (sharegpt: ilk 'human' turu)
def soruyu_al(ornek):
    return ornek["conversations"][0]["value"]

# Beklenen cagri isimleri: ikinci turdaki <tool_call>'lardan
def beklenen_araclar(ornek):
    turlar = ornek["conversations"]
    if len(turlar) < 2:
        return []
    cagrilar, _ = cagrilari_ayikla(turlar[1]["value"])
    return [c.get("name") for c in cagrilar]

# Ornek kontrol
_o = gorulmemis[0]
print("SORU     :", soruyu_al(_o)[:150])
print("BEKLENEN :", beklenen_araclar(_o))

In [ ]:
def matematik_calistir(model_adi: str) -> dict:
    n = len(gorulmemis)
    format_ok = arac_ok = abstain_ok = 0
    abstain_toplam = cagri_toplam = 0
    kayitlar = []
    basla = time.time()

    with model_olarak(model_adi):
        for i in range(n):
            ornek = gorulmemis[i]
            soru = soruyu_al(ornek)
            beklenen = beklenen_araclar(ornek)

            # max_new_tokens=256: <think> + <tool_call> JSON'u sigacak kadar
            cikti = uret(soru, max_new_tokens=256)
            uretilen, gecerli = cagrilari_ayikla(cikti)
            uretilen_isimler = [c.get("name") for c in uretilen]

            if gecerli:
                format_ok += 1

            if beklenen:                      # arac CAGRILMALI olan ornek
                cagri_toplam += 1
                # kume karsilastirmasi: sira onemli degil, dogru araclar cagrildi mi
                if set(uretilen_isimler) == set(beklenen):
                    arac_ok += 1
            else:                             # arac cagrilMAMALI olan ornek
                abstain_toplam += 1
                if not uretilen_isimler:
                    abstain_ok += 1

            kayitlar.append({
                "no": i, "model": model_adi,
                "beklenen": beklenen, "uretilen": uretilen_isimler,
                "format_gecerli": gecerli,
                "cikti": cikti[:300],
            })

            skor = (arac_ok + abstain_ok) / (i + 1) * 100
            print(f"\r[{model_adi:9}] {ilerleme(i+1, n)} genel {skor:5.1f}% "
                  f"({time.time()-basla:.0f}s)", end="")
    print()

    return {
        "model": model_adi,
        "format_gecerlilik": round(100 * format_ok / n, 2),
        "arac_secim_dogrulugu": round(100 * arac_ok / cagri_toplam, 2) if cagri_toplam else None,
        "abstain_dogrulugu": round(100 * abstain_ok / abstain_toplam, 2) if abstain_toplam else None,
        "genel_dogruluk": round(100 * (arac_ok + abstain_ok) / n, 2),
        "cagri_gereken": cagri_toplam,
        "cagri_gerekmeyen": abstain_toplam,
        "sure_sn": round(time.time() - basla, 1),
        "kayitlar": kayitlar,
    }


mat_sonuc = {ad: matematik_calistir(ad) for ad in ("base", "finetune")}
for ad, s in mat_sonuc.items():
    print(f"{ad:9} -> genel %{s['genel_dogruluk']} | arac %{s['arac_secim_dogrulugu']} | "
          f"abstain %{s['abstain_dogrulugu']} | format %{s['format_gecerlilik']}")

## 7. Karşılaştırma

In [ ]:
def fark(yeni, eski):
    if yeni is None or eski is None:
        return "-"
    d = yeni - eski
    return f"{d:+.2f}"

ozet = pd.DataFrame([
    {"Benchmark": "MMLU (genel bilgi)", "Metrik": "Başarı %",
     "Base": mmlu_sonuc['base']['basari'], "Fine-tune": mmlu_sonuc['finetune']['basari'],
     "Fark": fark(mmlu_sonuc['finetune']['basari'], mmlu_sonuc['base']['basari'])},
    {"Benchmark": "Matematik Tool-Call", "Metrik": "Genel doğruluk %",
     "Base": mat_sonuc['base']['genel_dogruluk'], "Fine-tune": mat_sonuc['finetune']['genel_dogruluk'],
     "Fark": fark(mat_sonuc['finetune']['genel_dogruluk'], mat_sonuc['base']['genel_dogruluk'])},
    {"Benchmark": "Matematik Tool-Call", "Metrik": "Araç seçimi %",
     "Base": mat_sonuc['base']['arac_secim_dogrulugu'], "Fine-tune": mat_sonuc['finetune']['arac_secim_dogrulugu'],
     "Fark": fark(mat_sonuc['finetune']['arac_secim_dogrulugu'], mat_sonuc['base']['arac_secim_dogrulugu'])},
    {"Benchmark": "Matematik Tool-Call", "Metrik": "Çekimserlik (abstain) %",
     "Base": mat_sonuc['base']['abstain_dogrulugu'], "Fine-tune": mat_sonuc['finetune']['abstain_dogrulugu'],
     "Fark": fark(mat_sonuc['finetune']['abstain_dogrulugu'], mat_sonuc['base']['abstain_dogrulugu'])},
    {"Benchmark": "Matematik Tool-Call", "Metrik": "Format geçerliliği %",
     "Base": mat_sonuc['base']['format_gecerlilik'], "Fine-tune": mat_sonuc['finetune']['format_gecerlilik'],
     "Fark": fark(mat_sonuc['finetune']['format_gecerlilik'], mat_sonuc['base']['format_gecerlilik'])},
])
display(ozet)

print("\nNASIL OKUNUR")
print("  MMLU farki ~0 veya pozitif  -> genel yetenek korunmus (istedigimiz bu)")
print("  MMLU farki cok negatif      -> asiri uyum, model daralmis")
print("  Matematik farki pozitif     -> fine-tune ise yaramis")

In [ ]:
import matplotlib.pyplot as plt

metrikler = ozet['Metrik'].tolist()
base_d = [0 if v is None else v for v in ozet['Base']]
ft_d   = [0 if v is None else v for v in ozet['Fine-tune']]

x = range(len(metrikler)); g = 0.38
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar([i - g/2 for i in x], base_d, g, label='Base', color='#94a3b8')
ax.bar([i + g/2 for i in x], ft_d,   g, label='Fine-tune', color='#2563eb')

for i, (b, f) in enumerate(zip(base_d, ft_d)):
    ax.text(i - g/2, b + 1, f"{b:.1f}", ha='center', fontsize=9)
    ax.text(i + g/2, f + 1, f"{f:.1f}", ha='center', fontsize=9)

ax.set_xticks(list(x)); ax.set_xticklabels(metrikler, rotation=18, ha='right')
ax.set_ylabel('%'); ax.set_ylim(0, 105)
ax.set_title('Base vs Fine-tune')
ax.legend(); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

### Örnek çıktılar

Sayılar *ne kadar* değiştiğini söyler; asıl anlamak için çıktılara bakmak gerekir.

In [ ]:
# Fine-tune'un DOGRU, base'in YANLIS yaptigi ornekler -> fine-tune tam olarak ne kazandirdi
b_kayit = {k['no']: k for k in mat_sonuc['base']['kayitlar']}
f_kayit = {k['no']: k for k in mat_sonuc['finetune']['kayitlar']}

gosterildi = 0
for no, f in f_kayit.items():
    b = b_kayit[no]
    f_dogru = set(f['uretilen']) == set(f['beklenen'])
    b_dogru = set(b['uretilen']) == set(b['beklenen'])
    if f_dogru and not b_dogru:
        print("=" * 78)
        print("BEKLENEN :", f['beklenen'])
        print("BASE     :", b['uretilen'], "|", b['cikti'][:180].replace("\n", " "))
        print("FINETUNE :", f['uretilen'], "|", f['cikti'][:180].replace("\n", " "))
        gosterildi += 1
        if gosterildi >= 3:
            break

if gosterildi == 0:
    print("Fine-tune'un base'e ustunluk sagladigi ornek bulunamadi.")

In [ ]:
# Aksi yon: fine-tune'un BOZDUGU ornekler (varsa) -> durust degerlendirme icin sart
gosterildi = 0
for no, f in f_kayit.items():
    b = b_kayit[no]
    f_dogru = set(f['uretilen']) == set(f['beklenen'])
    b_dogru = set(b['uretilen']) == set(b['beklenen'])
    if b_dogru and not f_dogru:
        print("=" * 78)
        print("BEKLENEN :", f['beklenen'])
        print("BASE     :", b['uretilen'], "(dogru)")
        print("FINETUNE :", f['uretilen'], "(yanlis) |", f['cikti'][:180].replace("\n", " "))
        gosterildi += 1
        if gosterildi >= 3:
            break

if gosterildi == 0:
    print("Fine-tune hicbir ornegi bozmamis.")

## 8. Sonuçları kaydet

In [ ]:
# Ozet tablo
ozet.to_csv("benchmark_ozet.csv", index=False)

# Tum ham kayitlar (hangi soruya ne cevap verildi)
tum = []
for kaynak, sonuclar in (("mmlu", mmlu_sonuc), ("matematik", mat_sonuc)):
    for ad, s in sonuclar.items():
        for k in s["kayitlar"]:
            tum.append({"benchmark": kaynak, **k})
pd.DataFrame(tum).to_csv("benchmark_detay.csv", index=False)

# Bolum bazli MMLU karsilastirmasi
bolum_df = pd.DataFrame({
    "base":     mmlu_sonuc['base']['bolum_basari'],
    "finetune": mmlu_sonuc['finetune']['bolum_basari'],
}).fillna(0)
bolum_df["fark"] = bolum_df["finetune"] - bolum_df["base"]
bolum_df = bolum_df.sort_values("fark")
bolum_df.to_csv("benchmark_mmlu_bolum.csv")

print("Kaydedildi: benchmark_ozet.csv, benchmark_detay.csv, benchmark_mmlu_bolum.csv")
print("\nMMLU'da en cok DUSEN 5 bolum:"); display(bolum_df.head(5))
print("MMLU'da en cok YUKSELEN 5 bolum:"); display(bolum_df.tail(5))

---

### Not: `alibayram/*` veri setlerine yazma yok

Orijinal kodda sonuçlar `alibayram/...` repolarına `push_to_hub` ile gönderiliyordu.
O repolar **başkasına ait** — oraya yazmak doğru olmaz. Bu notebook sonuçları yalnızca
yerel CSV olarak kaydeder. Kendi hesabına yüklemek istersen:

```python
from datasets import Dataset
Dataset.from_pandas(ozet).push_to_hub("bilalabic/math-toolcall-benchmark", token=HF_TOKEN)
```

### Sonuçları nasıl yorumlamalı

- **MMLU'da büyük düşüş** → model dar alana aşırı uyum sağlamış. Çare: daha az epoch,
  daha düşük learning rate, ya da veriye genel amaçlı örnekler karıştırmak.
- **Matematikte kazanç yok** → LoRA `r=8` yetersiz kalmış olabilir; `r=16` deneyebilir
  ya da veri setini büyütebilirsin.
- **Format geçerliliği %100'e yakın ama araç seçimi düşük** → model formatı öğrenmiş
  ama *hangi* aracı seçeceğini öğrenememiş. Çeldirici araçlar fazla zorlayıcı olabilir.